# EDA 01 - データ全体概要

ATMA Cup 20 (Udemy) の全データセットに対する探索的データ分析。

**評価指標**: ROC-AUC  
**主キー**: `['社員番号', 'category']`  
**目的**: 各データの特性把握・品質確認・モデリング方針の検討

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

# 日本語フォント設定
matplotlib.rcParams['font.family'] = 'IPAexGothic'

# パス設定
DATA_DIR = Path("../data/raw/input")
FIG_DIR = Path("../data/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# データ読み込み
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")
df_udemy = pd.read_csv(DATA_DIR / "udemy_activity.csv")
df_career = pd.read_csv(DATA_DIR / "career.csv")
df_dx = pd.read_csv(DATA_DIR / "dx.csv")
df_hr = pd.read_csv(DATA_DIR / "hr.csv")
df_overtime = pd.read_csv(DATA_DIR / "overtime_work_by_month.csv")
df_position = pd.read_csv(DATA_DIR / "position_history.csv")

print("データ読み込み完了")

## 1. 基本確認 - train / test

In [ ]:
# 行数・列数
print("=== train ===")
print(f"shape: {df_train.shape}")
display(df_train.head(3))
display(df_train.dtypes.to_frame("dtype"))

print("\n=== test ===")
print(f"shape: {df_test.shape}")
display(df_test.head(3))

In [ ]:
# 欠損値・重複確認
def check_basic(df, name):
    print(f"=== {name} ===")
    null_rate = df.isnull().mean().rename("null_rate")
    print(null_rate[null_rate > 0].to_string() or "  欠損なし")
    print(f"  重複行数: {df.duplicated().sum()}")
    print()

check_basic(df_train, "train")
check_basic(df_test, "test")

## 2. 目的変数の分析

In [ ]:
# 目的変数の列を自動検定（train にあって test にない列）
list_target_cols = [c for c in df_train.columns if c not in df_test.columns and c not in ["社員番号", "category"]]
print(f"目的変数候補: {list_target_cols}")
target_col = list_target_cols[0]

# クラス比率
print(f"\n=== {target_col} の分布 ===")
vc = df_train[target_col].value_counts()
print(vc)
print(f"\nPositive rate: {vc.get(1, 0) / len(df_train):.3f}")

# category別の目的変数分布
if "category" in df_train.columns:
    print("\n=== category別 Positive rate ===")
    display(df_train.groupby("category")[target_col].mean().to_frame("positive_rate"))

In [ ]:
# 目的変数の可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 全体分布
df_train[target_col].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "salmon"])
axes[0].set_title(f"{target_col} の分布（全体）")
axes[0].set_xlabel(target_col)
axes[0].set_ylabel("件数")

# category別
if "category" in df_train.columns:
    df_train.groupby("category")[target_col].mean().plot(kind="bar", ax=axes[1], color="steelblue")
    axes[1].set_title("category別 Positive rate")
    axes[1].set_xlabel("category")
    axes[1].set_ylabel("Positive rate")
    axes[1].axhline(df_train[target_col].mean(), color="red", linestyle="--", label="全体平均")
    axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_target_distribution.png", dpi=120)
plt.show()

## 3. 補助テーブルの基本確認

In [ ]:
dict_sub_tables = {
    "udemy_activity": df_udemy,
    "career": df_career,
    "dx": df_dx,
    "hr": df_hr,
    "overtime": df_overtime,
    "position_history": df_position,
}

# 各テーブルの概要を一覧表示
list_summary = []
for name, df in dict_sub_tables.items():
    list_summary.append({
        "テーブル": name,
        "行数": df.shape[0],
        "列数": df.shape[1],
        "社員番号数": df["社員番号"].nunique() if "社員番号" in df.columns else None,
        "欠損列数": (df.isnull().sum() > 0).sum(),
    })

display(pd.DataFrame(list_summary))

In [ ]:
# 各テーブルのカバレッジ確認（train/testの社員番号に対するカバー率）
set_train_ids = set(df_train["社員番号"].unique())
set_test_ids = set(df_test["社員番号"].unique())
set_all_ids = set_train_ids | set_test_ids

print(f"train 社員番号数: {len(set_train_ids)}")
print(f"test  社員番号数: {len(set_test_ids)}")
print(f"重複（train & test）: {len(set_train_ids & set_test_ids)}")
print()

for name, df in dict_sub_tables.items():
    if "社員番号" not in df.columns:
        continue
    set_ids = set(df["社員番号"].unique())
    coverage_train = len(set_ids & set_train_ids) / len(set_train_ids)
    coverage_test = len(set_ids & set_test_ids) / len(set_test_ids)
    print(f"{name:20s} | train coverage: {coverage_train:.3f} | test coverage: {coverage_test:.3f}")

## 4. 各テーブルの詳細確認

In [ ]:
## udemy_activity: 受講ログ
print("=== udemy_activity ===")
display(df_udemy.head(3))
display(df_udemy.describe(include="all").T)

# 社員あたり受講数の分布
s_udemy_count = df_udemy.groupby("社員番号").size()
print(f"\n社員あたり受講レコード数: mean={s_udemy_count.mean():.1f}, median={s_udemy_count.median():.0f}, max={s_udemy_count.max()}")

In [ ]:
## overtime: 残業時間
print("=== overtime_work_by_month ===")
display(df_overtime.head(3))
display(df_overtime.describe())

# 時系列範囲
df_overtime["date"] = pd.to_datetime(df_overtime["date"])
print(f"\n期間: {df_overtime['date'].min()} 〜 {df_overtime['date'].max()}")

# 残業時間の分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_overtime["hours"].hist(bins=50, ax=axes[0])
axes[0].set_title("残業時間の分布")
axes[0].set_xlabel("hours")

df_overtime.groupby("date")["hours"].mean().plot(ax=axes[1])
axes[1].set_title("月別 平均残業時間の推移")
axes[1].set_xlabel("date")
axes[1].set_ylabel("平均残業時間")

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_overtime.png", dpi=120)
plt.show()

In [ ]:
## position_history: 職位履歴
print("=== position_history ===")
display(df_position.head(3))

print("\n勤務区分の種類:")
print(df_position["勤務区分"].value_counts())

print("\n役職の種類（上位20件）:")
print(df_position["役職"].value_counts().head(20))

# 社員あたりの履歴年数
s_pos_years = df_position.groupby("社員番号")["year"].nunique()
print(f"\n社員あたり記録年数: mean={s_pos_years.mean():.1f}, max={s_pos_years.max()}")

In [ ]:
## career: キャリア感アンケート
print("=== career ===")
display(df_career.head(3))
print(f"\nshape: {df_career.shape}")
check_basic(df_career, "career")

In [ ]:
## dx・hr: 研修受講記録
print("=== dx ===")
display(df_dx.head(3))
print(f"研修カテゴリ: {df_dx['研修カテゴリ'].nunique()}種類")
print(df_dx["研修カテゴリ"].value_counts().head(10))

print("\n=== hr ===")
display(df_hr.head(3))
print(f"カテゴリ: {df_hr['カテゴリ'].nunique()}種類")
print(df_hr["カテゴリ"].value_counts().head(10))

## 5. 所感・モデリングへの示唆

※ ノートブック実行後にここに記入する

### データ品質
- 

### 特徴量設計のヒント
- 

### 注意点・リスク
- 